# Download and Data Preparation

The following notebook is recommended to be run prior to running the main optimization_engine.ipynb to grab new data and prepare it to be processed by the optimization engine.

The existing processed data as of this point was made by this script on 11/14/2026.

## Dependencies Setup

Run all the cells below in order. The notebook installs missing packages, downloads the public data, and creates the data files.

In [1]:
import sys
import subprocess
import importlib.util
from pathlib import Path

packages = {
    "pandas": "pandas>=2.0.0",
    "numpy": "numpy>=1.24.0",
    "matplotlib": "matplotlib>=3.7.0",
    "requests": "requests>=2.31.0",
    "lxml": "lxml>=4.9.0"
}

for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

from itertools import combinations
from io import StringIO
from datetime import date
from IPython.display import display


data_folder = Path("data")
image_folder = Path("images")

data_folder.mkdir(exist_ok=True)
image_folder.mkdir(exist_ok=True)

## Download Alibaba Microservices v2022

I use the first 30-minute resource archive and the first three-minute response-time archive. The notebook downloads only these files, not the full trace. The normalized CPU, memory and call-rate values are not physical resource counts or requests per second.

In [2]:
import gzip
import tarfile
import json
import time
import re
from lxml import html
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

raw_folder = data_folder / "raw"
raw_folder.mkdir(exist_ok=True)

session = requests.Session()
retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retry))

alibaba_url = "https://github.com/alibaba/clusterdata/tree/master/cluster-trace-microservices-v2022"
archive_url = "https://aliopentrace.oss-cn-beijing.aliyuncs.com/v2022MicroservicesTraces/"
archive_paths = ["MSMetricsUpdate/MSMetricsUpdate_0.tar.gz", "MCRRTUpdate/MCRRTUpdate_0.tar.gz"]


def download_file(url, path):
    if path.exists() and path.stat().st_size > 0:
        print("Using existing file:", path.name)
        return

    temporary = path.with_suffix(path.suffix + ".part")
    print("Downloading:", path.name, flush=True)

    with session.get(url, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        with temporary.open("wb") as file:
            for block in response.iter_content(chunk_size=1024 * 1024):
                file.write(block)

    temporary.replace(path)


for archive_path in archive_paths:
    download_file(archive_url + archive_path, raw_folder / Path(archive_path).name)

Downloading: MSMetricsUpdate_0.tar.gz


Downloading: MCRRTUpdate_0.tar.gz


## Process the Alibaba Microservices v2022 Data

The archives contain a lot of unneeded data for our use-case, so the following script reduces just to what we need.

In [3]:
# The column order comes from Alibaba's README.
keys = ["timestamp", "msname", "msinstanceid", "nodeid"]
resource_columns = keys + ["cpu_utilization", "memory_utilization"]
metric_columns = keys + [
    "providerrpc_rt", "providerrpc_mcr", "consumerrpc_rt", "consumerrpc_mcr",
    "writemc_rt", "writemc_mcr", "readmc_rt", "readmc_mcr",
    "writedb_rt", "writedb_mcr", "readdb_rt", "readdb_mcr",
    "consumermq_rt", "consumermq_mcr", "providermq_rt", "providermq_mcr",
    "http_mcr", "http_rt"
]


def read_archive(path, columns):
    # Read in chunks so the entire trace does not need to fit in memory.
    with tarfile.open(path, "r:gz") as archive:
        for member in archive:
            if not member.isfile() or Path(member.name).name.startswith("."):
                continue

            with archive.extractfile(member) as file:
                stream = gzip.GzipFile(fileobj=file) if member.name.endswith(".gz") else file
                for chunk in pd.read_csv(stream, header=None, names=columns,
                                         dtype=str, chunksize=100000):
                    timestamps = pd.to_numeric(chunk["timestamp"], errors="coerce")
                    chunk = chunk.loc[timestamps.notna()].copy()
                    chunk["timestamp"] = timestamps.loc[chunk.index].astype("int64")
                    chunk = chunk.dropna(subset=keys)
                    chunk = chunk[chunk["msname"].str.startswith("MS_")
                                  & chunk["nodeid"].str.startswith("NODE_")]
                    yield chunk


# Sample valid provider RPC records from the first archive.
parts = []
count = 0

for chunk in read_archive(raw_folder / "MCRRTUpdate_0.tar.gz", metric_columns):
    for column in ["providerrpc_rt", "providerrpc_mcr"]:
        chunk[column] = pd.to_numeric(chunk[column], errors="coerce")

    valid = chunk[(chunk["providerrpc_rt"] > 0) & (chunk["providerrpc_mcr"] > 0)
                  & (chunk["providerrpc_mcr"] <= 1)]
    parts.append(valid)
    count += len(valid)

    if count >= 150000:
        break

if not parts:
    raise RuntimeError("No Alibaba response-time records found. Check the archive.")

metrics = pd.concat(parts, ignore_index=True).head(150000)
metrics.to_csv(data_folder / "original_alibaba_metrics_sample.csv", index=False)

# Average duplicate measurements instead of multiplying rows during the join.
metrics_clean = metrics.groupby(keys, as_index=False)[["providerrpc_rt", "providerrpc_mcr"]].mean()
resource_parts = []

for chunk in read_archive(raw_folder / "MSMetricsUpdate_0.tar.gz", resource_columns):
    matches = chunk.merge(metrics_clean[keys], on=keys, how="inner")
    if not matches.empty:
        resource_parts.append(matches)

if not resource_parts:
    raise RuntimeError("No matching Alibaba records. Check timestamps and archive contents.")

resources = pd.concat(resource_parts, ignore_index=True)
resources.to_csv(data_folder / "original_alibaba_resource_sample.csv", index=False)

for column in ["cpu_utilization", "memory_utilization"]:
    resources[column] = pd.to_numeric(resources[column], errors="coerce")

resources = resources[resources["cpu_utilization"].between(0, 1)
                      & resources["memory_utilization"].between(0, 1)]
resources_clean = resources.groupby(keys, as_index=False)[["cpu_utilization", "memory_utilization"]].mean()

joined = metrics_clean.merge(resources_clean, on=keys, validate="one_to_one")
joined = joined.replace([np.inf, -np.inf], np.nan).dropna()
joined.to_csv(data_folder / "alibaba_joined.csv", index=False)

if joined["msname"].nunique() < 6:
    raise RuntimeError("Not enough distinct services matched. Inspect the original samples.")

print("Matched records:", len(joined))
print("Distinct services:", joined["msname"].nunique())
display(joined.head())

Matched records: 11959
Distinct services: 4214


,timestamp,msname,msinstanceid,nodeid,providerrpc_rt,providerrpc_mcr,cpu_utilization,memory_utilization
0,0,MS_10025,MS_10025_POD_52,NODE_29161,124.514563,3.034569e-06,0.120409,0.657554
1,0,MS_10025,MS_10025_POD_77,NODE_27651,145.019802,2.975646e-06,0.148975,0.674028
2,0,MS_10026,MS_10026_POD_11,NODE_26482,3.772036,6.461963e-06,0.037432,0.436981
3,0,MS_10042,MS_10042_POD_9,NODE_22628,584.800000,4.910306e-08,0.029340,0.483108
4,0,MS_10077,MS_10077_POD_17,NODE_40145,3.257154,1.211373e-04,0.198450,0.634625


## Download RIPE Atlas Measurements

I use public IPv4 anchor-mesh ping results from five geographic locations. The source and target anchors are different even within the same location. Missing routes are not filled in.

In [4]:
ripe_url = "https://atlas.ripe.net/api/v2/"
regions = {
    "US East": ("US", ["New York", "Newark"]),
    "US West": ("US", ["San Francisco", "San Jose", "Los Angeles"]),
    "London": ("GB", ["London"]),
    "Singapore": ("SG", ["Singapore"]),
    "Tokyo": ("JP", ["Tokyo"])
}


def get_json(url, params=None):
    response = session.get(url, params=params, timeout=(30, 120))
    response.raise_for_status()
    return response.json()


def get_pages(url, params):
    while url:
        page = get_json(url, params)
        yield from page["results"]
        url = page.get("next")
        params = None


# Save the selected endpoints and time window for reproducibility.
ripe_cache = raw_folder / "ripe_selection.json"

if ripe_cache.exists():
    selection = json.loads(ripe_cache.read_text(encoding="utf-8"))
else:
    anchors = {}
    for region, (country, cities) in regions.items():
        candidates = list(get_pages(ripe_url + "anchors/", {
            "country": country, "is_active": "true", "probe_status": 1, "page_size": 100
        }))
        candidates = [a for a in candidates if a.get("probe") and a.get("ip_v4")
                      and any(city.casefold() in a.get("city", "").casefold() for city in cities)]
        candidates.sort(key=lambda a: a["id"])
        if len(candidates) < 2:
            raise RuntimeError(f"Need two active RIPE anchors in {region}. Check the region list.")
        anchors[region] = candidates[:4]

    stop = int(time.time()) - 3600
    selection = {"anchors": anchors, "start": stop - 86400, "stop": stop}
    ripe_cache.write_text(json.dumps(selection, indent=2), encoding="utf-8")

anchors = selection["anchors"]
source_probes = {a["probe"]: region for region, candidates in anchors.items() for a in candidates}
rows = []

for target_region, candidates in anchors.items():
    # Try other anchors in the same region if a target lacks measurements.
    covered = set()

    for target in candidates:
        query = {"tags": "anchoring,mesh", "type": "ping", "af": 4,
                 "status": 2, "target_ip": target["ip_v4"], "page_size": 50}
        measurements = list(get_pages(ripe_url + "measurements/", query))
        if not measurements:
            continue

        measurement_id = measurements[0]["id"]
        cache = raw_folder / f"ripe_{measurement_id}.json"

        if cache.exists():
            results = json.loads(cache.read_text(encoding="utf-8"))
        else:
            results = get_json(ripe_url + f"measurements/{measurement_id}/results/", {
                "start": selection["start"], "stop": selection["stop"],
                "probe_ids": ",".join(map(str, source_probes))
            })
            cache.write_text(json.dumps(results), encoding="utf-8")

        for result in results:
            probe_id = result.get("prb_id")
            if probe_id not in source_probes or probe_id == target["probe"]:
                continue

            for packet in result.get("result", []):
                rtt = packet.get("rtt")
                if isinstance(rtt, (int, float)) and np.isfinite(rtt) and rtt > 0:
                    source_region = source_probes[probe_id]
                    covered.add(source_region)
                    rows.append({"source_region": source_region, "target_region": target_region,
                                 "probe_id": probe_id, "target_anchor_id": target["id"],
                                 "measurement_id": measurement_id, "timestamp": result["timestamp"],
                                 "rtt_ms": rtt})

        if covered == set(regions):
            break

    print(target_region, "source regions measured:", len(covered))

if not rows:
    raise RuntimeError("RIPE Atlas returned no successful pings.")

network = pd.DataFrame(rows)
network.to_csv(data_folder / "original_ripe_ping_sample.csv", index=False)
network_summary = network.groupby(["source_region", "target_region"])["rtt_ms"].agg(
    latency_ms="median", packet_count="count").reset_index()
network_summary.to_csv(data_folder / "ripe_region_rtt.csv", index=False)

latency_matrix = network_summary.pivot(index="source_region", columns="target_region", values="latency_ms")
latency_matrix = latency_matrix.reindex(index=list(regions), columns=list(regions))

if latency_matrix.isna().any().any():
    display(latency_matrix)
    raise RuntimeError("Incomplete RIPE RTT matrix. Review the saved measurements before continuing.")

display(latency_matrix.round(2))

US East source regions measured: 5


US West source regions measured: 5


London source regions measured: 5


Singapore source regions measured: 5


Tokyo source regions measured: 5


target_region,US East,US West,London,Singapore,Tokyo
source_region,,,,,
US East,2.38,71.06,69.02,214.80,166.70
US West,68.07,11.97,136.75,173.40,114.47
London,70.85,141.37,1.85,164.67,230.01
Singapore,223.30,172.34,157.58,1.05,76.38
Tokyo,157.32,107.89,232.84,69.99,1.93


## Cloudflare Pricing

I use Cloudflare's published container rates and instance sizes as the cost baseline. These are customer prices, which are not Cloudflare's internal costs, so the prices are higher but still illustrates and shows the impact on infrastructure.

In [5]:
pricing_url = "https://developers.cloudflare.com/containers/platform/pricing/"

pricing_response = session.get(pricing_url, timeout=(30, 120))
pricing_response.raise_for_status()
(raw_folder / "cloudflare_pricing.html").write_bytes(pricing_response.content)


def download_pricing():
    import re
    from lxml import html

    response = pricing_response

    text = " ".join(html.fromstring(response.content).text_content().split())
    rows = []

    for parameter, unit in [
        ("cpu_vcpu_second", "vCPU"),
        ("memory_gib_second", "GiB"),
        ("disk_gb_second", "GB")
    ]:
        pattern = r"\$([0-9.]+)\s*per additional\s+" + unit + r"-second"
        match = re.search(pattern, text, re.IGNORECASE)

        if match is None:
            raise ValueError(f"Could not find the {unit} rate.")

        rows.append({
            "parameter": parameter,
            "rate": float(match.group(1)),
            "unit": f"USD/{unit}-second",
            "source": pricing_url,
            "accessed": date.today().isoformat()
        })

    return pd.DataFrame(rows)


def download_instances():
    response = pricing_response

    tables = pd.read_html(StringIO(response.text))

    for table in tables:
        if not {"Instance Type", "vCPU", "Memory", "Disk"}.issubset(table.columns):
            continue

        rows = []

        for _, row in table.iterrows():
            cpu = str(row["vCPU"]).strip()

            if "/" in cpu:
                numerator, denominator = cpu.split("/")
                cpu = float(numerator) / float(denominator)
            else:
                cpu = float(cpu)

            memory, unit = str(row["Memory"]).split()
            memory = float(memory)

            if unit == "MiB":
                memory = memory / 1024

            disk = float(str(row["Disk"]).split()[0])

            rows.append({
                "instance_type": str(row["Instance Type"]),
                "vcpu": cpu,
                "memory_gib": memory,
                "disk_gb": disk,
                "source": pricing_url,
                "accessed": date.today().isoformat()
            })

        return pd.DataFrame(rows)

    raise ValueError("Could not locate the instance types.")


def load_data(filename, download):
    data = download()
    data.to_csv(data_folder / filename, index=False)
    print(f"Downloaded: {filename}")

    return data


pricing = load_data("original_cloudflare_pricing.csv", download_pricing)
instances = load_data("original_cloudflare_instances.csv", download_instances)

display(pricing)
display(instances)

assert pricing["parameter"].is_unique and (pricing["rate"] > 0).all()
assert instances["instance_type"].is_unique

Downloaded: original_cloudflare_pricing.csv
Downloaded: original_cloudflare_instances.csv


,parameter,rate,unit,source,accessed
0,cpu_vcpu_second,2.000000e-05,USD/vCPU-second,https://developers.cloudflare.com/containers/p...,2026-11-14
1,memory_gib_second,2.500000e-06,USD/GiB-second,https://developers.cloudflare.com/containers/p...,2026-11-14
2,disk_gb_second,7.000000e-08,USD/GB-second,https://developers.cloudflare.com/containers/p...,2026-11-14


,instance_type,vcpu,memory_gib,disk_gb,source,accessed
0,lite,0.0625,0.25,2.0,https://developers.cloudflare.com/containers/p...,2026-11-14
1,basic,0.2500,1.00,4.0,https://developers.cloudflare.com/containers/p...,2026-11-14
2,standard-1,0.5000,4.00,8.0,https://developers.cloudflare.com/containers/p...,2026-11-14
3,standard-2,1.0000,6.00,12.0,https://developers.cloudflare.com/containers/p...,2026-11-14
4,standard-3,2.0000,8.00,16.0,https://developers.cloudflare.com/containers/p...,2026-11-14
5,standard-4,4.0000,12.00,20.0,https://developers.cloudflare.com/containers/p...,2026-11-14


## Next Step

After all the cells above finish, the processed data is saved in the `data/` folder. Run `optimization_engine.ipynb` next.